# Synthetic Regime-Switch Experiment

End-to-end notebook for the synthetic experiment in Section 4.1 of the manuscript. The flow:

1. Clone the repo and set up the `uv` environment.
2. Generate the regime-switch parquet matching `src/data/dataset.py` schema.
3. Compute training-free baselines (random, weighted random, individual base models, oracle).
4. Train the proposed hierarchical transformer router.
5. Train the MLP-pool baseline router.
6. Evaluate both checkpoints on the same test split.
7. (Optional) Mirror the logs to Google Drive.

All cells use `!uv run python -m ...` so the notebook stays reproducible and matches the CLI used elsewhere in the project.

## 1. Initial setup (clone + uv env + secrets)

In [ ]:
! git clone https://github.com/huseyin-karaca/s2t-tr-dev
%cd /content/s2t-tr-dev

from google.colab import files
files.view('/content/s2t-tr-dev')

!make create_environment
!make requirements

from google.colab import userdata
import os
os.environ['GITHUB_TOKEN'] = userdata.get('GitHubPAT')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

!gh auth status
!git config --global user.name 'huseyin-karaca'
!git config --global user.email 'huseyinkaraccca@gmail.com'

## 2. Generate the synthetic regime-switch parquet

The output file follows the same schema as the real-world parquets: three frame-level feature columns and three per-expert WER columns. With the defaults (`N=10000`, `T=128`, `R=4`, `K=3`) the file is around 2 GB on disk because the embeddings dominate; reduce `--num-samples` for a quick check first.

In [ ]:
# Smoke run first — small file to confirm the pipeline before committing to the full size.
!uv run python -m src.data.synthetic \
    --output-path data/processed/synthetic_regime_switch_smoke/combined_features.parquet \
    --num-samples 200 --frame-length 32 --seed 42

In [ ]:
# Full synthetic dataset.
!uv run python -m src.data.synthetic \
    --output-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --num-samples 10000 --frame-length 128 \
    --num-regimes 4 --regime-dim 32 --noise-std 0.5 \
    --best-wer 0.15 --worst-wer 0.50 --wer-noise-std 0.02 \
    --seed 42

## 3. Training-free baselines

These do not require a learned model: oracle, random, weighted random, and each base model on its own. The script reports numbers on the same test split that the trained routers are evaluated on (matching `--seed`, `--train-ratio`, `--val-ratio`).

In [ ]:
!uv run python -m src.training.eval_baselines \
    --parquet-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --train-ratio 0.8 --val-ratio 0.1 --seed 42 --split test \
    --save-json logs/synthetic_regime_switch_baselines.json

## 4. Train the proposed hierarchical transformer router

Defaults match the manuscript: `d=256`, 2 Stage-1 layers, 1 Stage-2 layer, cross-attention bridge enabled, shared Stage-1 weights. The synthetic loss configuration uses weighted-WER plus soft cross-entropy, the best combination identified by the loss ablation (Section 4.3.1 of the manuscript).

In [ ]:
!uv run python -m src.training.train \
    --parquet-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --arch hierarchical_transformer \
    --max-seq-len 256 \
    --batch-size 32 --num-workers 2 \
    --d-model 256 --n-heads 4 \
    --stage1-layers 2 --stage2-layers 1 --ffn-dim 512 \
    --dropout 0.15 \
    --learning-rate 1e-4 --weight-decay 1e-2 --warmup-steps 200 \
    --max-epochs 50 \
    --primary-weight 1.0 --aux-ce-weight 0.0 \
    --soft-ce-weight 0.5 --soft-ce-temperature 0.1 \
    --seed 42 \
    --experiment-name synthetic_regime_switch_hier

## 5. Train the MLP-pool baseline router

Same loss and the same seed/splits as the proposed router; only the architecture changes. The MLP mean-pools each expert's frame sequence and feeds the concatenation through a 2-layer MLP.

In [ ]:
!uv run python -m src.training.train \
    --parquet-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --arch mlp_pool \
    --max-seq-len 256 \
    --batch-size 32 --num-workers 2 \
    --mlp-hidden 1024 --mlp-layers 2 \
    --dropout 0.15 \
    --learning-rate 1e-4 --weight-decay 1e-2 --warmup-steps 200 \
    --max-epochs 50 \
    --primary-weight 1.0 --aux-ce-weight 0.0 \
    --soft-ce-weight 0.5 --soft-ce-temperature 0.1 \
    --seed 42 \
    --experiment-name synthetic_regime_switch_mlp_pool

## 6. Evaluate the trained checkpoints on the test split

Both runs are evaluated on the same test split (same seed and split ratios). The training script already runs a final test pass automatically and dumps results to TensorBoard; the cells below repeat it as a standalone JSON snapshot for the manuscript table.

In [ ]:
!uv run python -m src.training.evaluate \
    --checkpoint logs/synthetic_regime_switch_hier/checkpoints/last.ckpt \
    --parquet-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --train-ratio 0.8 --val-ratio 0.1 --seed 42 --split test \
    --batch-size 32 --max-seq-len 256 --num-workers 2 \
    --save-json logs/synthetic_regime_switch_hier/test_results.json

In [ ]:
!uv run python -m src.training.evaluate \
    --checkpoint logs/synthetic_regime_switch_mlp_pool/checkpoints/last.ckpt \
    --parquet-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --train-ratio 0.8 --val-ratio 0.1 --seed 42 --split test \
    --batch-size 32 --max-seq-len 256 --num-workers 2 \
    --save-json logs/synthetic_regime_switch_mlp_pool/test_results.json

## 7. (Optional) Mirror logs and result JSONs to Google Drive

Useful for keeping a copy of the run artefacts after the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/gdrive')

In [ ]:
!mkdir -p "/gdrive/My Drive/s2t-tr-dev/logs"
!cp -r logs/synthetic_regime_switch_hier      "/gdrive/My Drive/s2t-tr-dev/logs/"
!cp -r logs/synthetic_regime_switch_mlp_pool   "/gdrive/My Drive/s2t-tr-dev/logs/"
!cp     logs/synthetic_regime_switch_baselines.json "/gdrive/My Drive/s2t-tr-dev/logs/"

## 8. (Optional) Release the runtime

In [ ]:
from google.colab import runtime
runtime.unassign()